In [ ]:
%matplotlib inline

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import torch
import triton

import cute_vector_add
import triton_vector_add
import tilelang_vector_add
from common.asm import dump_asm
from common.inputs import make_inputs
from common.run import make_runner, run_benchmark
from neuroslop.visualize_bench_results import plot_benchmark

N = 1 << 20

In [ ]:
AUTOTUNE = True  # False runs each kernel with its own defaults

TRITON_KERNEL = triton_vector_add.vector_add_kernel(autotune=AUTOTUNE)

IMPLEMENTATIONS = {
    "torch": make_runner("torch", torch.add),
    "torch_compile": make_runner("torch_compile", lambda x, y: x + y),
    "triton": make_runner(
        "triton",
        TRITON_KERNEL,
        extra_args=lambda a, b, out: (a.numel(),),
        grid=lambda meta: (triton.cdiv(meta["n"], meta["BLOCK_SIZE"]),),
    ),
    "tilelang": make_runner("tilelang", tilelang_vector_add.add_, specialize=lambda kernel, a, b: kernel(a.numel()), name="vector_add"),
    "cute": make_runner("cute", cute_vector_add.add(autotune=AUTOTUNE)),
    "cuda": make_runner("cuda", "cuda_vector_add.cu", entry="add"),
}

In [ ]:
# Tests: sizes around the block boundary catch kernels that drop the tail.
for n in [1, 255, 256, 257, 1000, N]:
    inputs = make_inputs((n,), (n,))
    for name, fn in IMPLEMENTATIONS.items():
        torch.testing.assert_close(fn(*inputs), torch.add(*inputs), msg=f"{name}, n={n}")
print("OK")

In [ ]:
inputs = make_inputs((N,), (N,))
results = run_benchmark(IMPLEMENTATIONS, inputs, reference=torch.add, flops=N)
plot_benchmark(results, baseline="torch_compile", title=f"{torch.cuda.get_device_name()} | N={N:,}")
plt.show()
if AUTOTUNE:
    print("triton config:", TRITON_KERNEL.best_config)
    print("cute config:", IMPLEMENTATIONS["cute"].asm()["add"]["config"])

In [ ]:
for name, fn in IMPLEMENTATIONS.items():
    dump_asm(fn.asm(), Path("asm") / name)
    print(name, list(fn.asm()))